In [ ]:
import os
import sys
import time
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sto usando il device: {device}")
if device == "cuda":
    print(f"Nome GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


repo_name = "BenchmarkingPathologyFoundationModels"
if not os.path.exists(repo_name):
    !git clone https://github.com/colin19950703/BenchmarkingPathologyFoundationModels.git
    print("Repository clonato con successo.")
else:
    print("Repository già presente.")


os.chdir(repo_name)
print(f"Directory di lavoro attuale: {os.getcwd()}")

/home/nelloconelli/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Sto usando il device: cuda
Nome GPU: NVIDIA GeForce RTX 4090 Laptop GPU
VRAM Totale: 16.73 GB
Repository già presente.
Directory di lavoro attuale: /home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/BenchmarkingPathologyFoundationModels


CTransPath

In [ ]:
import os
import requests
import subprocess
import sys


packages = ["loralib", "einops", "transformers", "peft", "tensorboard", "h5py", "openslide-python"]
print(f"   Installazione pacchetti extra: {', '.join(packages)}...")
subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)

repo_path = os.getcwd() # Siamo già dentro grazie alla Cella 1
pretrained_dir = os.path.join(repo_path, "model_lib", "pretrained")
os.makedirs(pretrained_dir, exist_ok=True)
weights_path = os.path.join(pretrained_dir, "ctranspath.pth")

url = "https://huggingface.co/jamesdolezal/CTransPath/resolve/main/ctranspath.pth"

if not os.path.exists(weights_path):
    try:
        response = requests.get(url, stream=True)
        with open(weights_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
    except Exception as e:
        print(f"Error downloading weights: {e}")
else:
    print("Weights Ctranspath already exist.")

🛠️ PASSAGGIO 3: Completamento Installazioni e Download Pesi
   Installazione pacchetti extra: loralib, einops, transformers, peft, tensorboard, h5py, openslide-python...
Defaulting to user installation because normal site-packages is not writeable
✅ Pesi CTransPath già presenti.


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm


SOURCE_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/Photos"

CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv"

OUTPUT_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/TILED_TIFF"

TILE_SIZE = 256    
STRIDE = 256       
THRESHOLD_WHITE = 230 

# ==========================================

def tile_microscopy_image(img_path, tile_size, stride, threshold_white):
    try:
        img = Image.open(img_path).convert('RGB')
        w, h = img.size
        
        patches = []
        

        for y in range(0, h - tile_size + 1, stride):
            for x in range(0, w - tile_size + 1, stride):

                patch = img.crop((x, y, x + tile_size, y + tile_size))
                

                gray = np.array(patch.convert('L'))
                white_pixels = np.sum(gray > threshold_white)
                total_pixels = tile_size * tile_size
                

                if (white_pixels / total_pixels) < 0.9:
                    patches.append((x, y, patch))
                    
        return patches
        
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return []

# --- MAIN ---
os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    df = pd.read_csv(CSV_PATH, header=None) 
    df.columns = ['filename', 'label'] 
    print(f"CSV uploaded: {len(df)} immagini.")
except:
    print("Error CSV")
    df = pd.DataFrame() 

class_map = {
    'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3,
    'normal': 0, 'benign': 1, 'insitu': 2, 'invasive': 3
}

print(f" TILING BACH -> TIFF ({TILE_SIZE}x{TILE_SIZE})...")
count = 0

for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    filename = row['filename']
    label_name = row['label']
    
    if not filename.endswith('.tif'):
        filename += '.tif'
        
    img_path = os.path.join(SOURCE_DIR, label_name, filename)
    
    if not os.path.exists(img_path):
        img_path = os.path.join(SOURCE_DIR, filename)
        
    if not os.path.exists(img_path):
        continue 
        
    class_idx = class_map.get(label_name, label_name) 
    save_dir = os.path.join(OUTPUT_DIR, str(class_idx))
    os.makedirs(save_dir, exist_ok=True)
    
    patches = tile_microscopy_image(img_path, TILE_SIZE, STRIDE, THRESHOLD_WHITE)
    
    base_name = os.path.splitext(filename)[0]
    for (x, y, patch) in patches:
        save_name = f"{base_name}_{x}_{y}.tiff" 
        

        patch.save(os.path.join(save_dir, save_name), "TIFF")
        
    count += 1



📖 CSV caricato: 400 immagini.
🚀 AVVIO TILING BACH -> TIFF (256x256)...


Processing: 100%|██████████| 400/400 [00:22<00:00, 17.97it/s]


✅ FINITO! Processate 400 immagini microscopiche in formato TIFF.


CTRANSPATH

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm
import timm
import gc
from sklearn.model_selection import train_test_split
import types
from sklearn.utils.class_weight import compute_class_weight
import requests
from huggingface_hub import login
import time  
import json  


gc.collect()
torch.cuda.empty_cache()



MODEL_TO_RUN = "ctranspath"   
BATCH_SIZE = 32            
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 4             # Normal, Benign, InSitu, Invasive

# PERCORSI
TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv"
SAVE_DIR = "./risultati_finali_bach"
CTRANSPATH_WEIGHTS = "./model_lib/pretrained/ctranspath.pth"


Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


def download_ctranspath_if_missing(path):
    if not os.path.exists(path):
        print(f"weiths not found in {path}. Downloading now...")
        os.makedirs(os.path.dirname(path), exist_ok=True)
        url = "https://huggingface.co/jamesdolezal/CTransPath/resolve/main/ctranspath.pth"
        try:
            response = requests.get(url, stream=True)
            with open(path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Download completed")
        except Exception as e:
            print(f"error download: {e}")
            raise e
    else:
        print(f"weights found in {path}.")

if MODEL_TO_RUN == "ctranspath":
    download_ctranspath_if_missing(CTRANSPATH_WEIGHTS)

# ==========================================

print(f"TRAINING ON BACH: {MODEL_TO_RUN.upper()}")


try:
    df = pd.read_csv(CSV_PATH, header=None, names=['filename', 'label_name'])
except:
    print("Error reading CSV.")


class_map = {'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3,
             'normal': 0, 'benign': 1, 'insitu': 2, 'invasive': 3}
df['label'] = df['label_name'].map(class_map)


df['base_name'] = df['filename'].apply(lambda x: os.path.splitext(x)[0])

base_names = df['base_name'].values
labels = df['label'].values


train_base_names, val_base_names = train_test_split(base_names, test_size=0.2, random_state=42, stratify=labels)

print(f"Dataset Split :")
print(f"   -> Train: {len(train_base_names)}")
print(f"   -> Val:   {len(val_base_names)}")


class BachTileDataset(Dataset):
    def __init__(self, root_dir, valid_base_names, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_set = set(valid_base_names)
        

        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            
            for img_name in os.listdir(class_path):
                if not img_name.endswith('.tiff'): continue
                
                parts = img_name.split('_')
                if len(parts) >= 3:
                    base_name = "_".join(parts[:-2])
                else:
                    base_name = os.path.splitext(img_name)[0]
                
                if base_name in valid_set:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder))
                    

    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            image = Image.open(path).convert('RGB')
            label = self.labels[idx]
            if self.transform: image = self.transform(image)
            return image, label
        except: return torch.zeros(3, 224, 224), 0

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = BachTileDataset(TILES_DIR, train_base_names, transform=train_transform)
val_dataset = BachTileDataset(TILES_DIR, val_base_names, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


print(f"Configuration Model: {MODEL_TO_RUN}...")


if MODEL_TO_RUN == "ctranspath":


    def to_2tuple(x): return tuple(x) if isinstance(x, (tuple, list)) else (x, x)

    class ConvStem(nn.Module):
        def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
            super().__init__()
            img_size = to_2tuple(img_size)
            patch_size = to_2tuple(patch_size)
            self.proj = nn.Sequential(
                nn.Conv2d(in_chans, embed_dim // 2, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim // 2), nn.ReLU(inplace=True),
                nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim), nn.ReLU(inplace=True),
            )
        def forward(self, x):
            x = self.proj(x)
            x = x.permute(0, 2, 3, 1) 
            return x

    model = timm.create_model(
        "swin_tiny_patch4_window7_224", 
        pretrained=False, 
        embed_dim=128,          
        depths=[2, 2, 18, 2],   
        num_heads=[4, 8, 16, 32]
    )
    model.patch_embed = ConvStem(img_size=224, patch_size=4, in_chans=3, embed_dim=128, norm_layer=nn.LayerNorm)

    if os.path.exists(CTRANSPATH_WEIGHTS):
        checkpoint = torch.load(CTRANSPATH_WEIGHTS, map_location="cpu")
        if 'model' in checkpoint: checkpoint = checkpoint['model']
        if 'state_dict' in checkpoint: checkpoint = checkpoint['state_dict']
        
        model_dict = model.state_dict()
        new_state_dict = {}
        
        for k, v in checkpoint.items():
            new_key = k
            new_key = new_key.replace('backbone.', '')
            
            if new_key == 'norm.weight' and 'norm.weight' not in model_dict: new_key = 'head.norm.weight'
            if new_key == 'norm.bias' and 'norm.bias' not in model_dict: new_key = 'head.norm.bias'
            
            if new_key in model_dict and v.shape == model_dict[new_key].shape:
                new_state_dict[new_key] = v
                
        model_dict.update(new_state_dict)
        msg = model.load_state_dict(model_dict, strict=False)
        
        del checkpoint
        del new_state_dict
        gc.collect()
        

        if len(msg.missing_keys) < 30:
            print("pretrained weights loaded successfully.")
        else:
            print(f"Warning: missing{len(msg.missing_keys)} pesi.")
    else:
        print("weights not found!")

    model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    
    def pooling_forward(self, x):
        x = self.forward_features(x)
        x = x.mean(dim=[1, 2]) 
        x = self.head(x)
        return x
    model.forward = types.MethodType(pooling_forward, model)


    for param in model.parameters(): 
        param.requires_grad = False
    for param in model.head.parameters(): 
        param.requires_grad = True

elif MODEL_TO_RUN == "phikon":
    model = timm.create_model("hf_hub:owkin/phikon", pretrained=True, num_classes=NUM_CLASSES)

elif MODEL_TO_RUN == "uni":
    model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, img_size=224, init_values=1e-5, num_classes=NUM_CLASSES)
    for param in model.parameters(): param.requires_grad = False
    for param in model.head.parameters(): param.requires_grad = True

elif MODEL_TO_RUN == "virchow2":
    model = timm.create_model("hf-hub:paige-ai/Virchow2", pretrained=True, 
                              mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU, num_classes=NUM_CLASSES)
    for param in model.parameters(): param.requires_grad = False
    for param in model.head.parameters(): param.requires_grad = True

model = model.to(device)



all_labels = train_dataset.labels 
if len(all_labels) > 0:
    cw = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)
    print(f"   -> Pesi: {class_weights}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

params_to_update = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(params_to_update, lr=LR)


history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []} 
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time() 

    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    epoch_time = time.time() - start_time 
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    

    print(f"Epoch {epoch+1}: Loss={epoch_loss:.4f} | Acc={val_acc:.2f}% | Time={epoch_time:.0f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(epoch_time)
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))


json_path = os.path.join(SAVE_DIR, f"stats_{MODEL_TO_RUN}.json")
with open(json_path, "w") as f:
    json.dump(history, f)
print(f"Statistic exported in: {json_path}")



PHYKON

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm
import timm
import gc
from sklearn.model_selection import train_test_split
import types
from sklearn.utils.class_weight import compute_class_weight
import requests 
from huggingface_hub import login


MODEL_TO_RUN = "phikon"   

BATCH_SIZE = 32            
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 4             

TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv"
SAVE_DIR = "./risultati_finali_bach"
CTRANSPATH_WEIGHTS = "./model_lib/pretrained/ctranspath.pth"


Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


def download_ctranspath_if_missing(path):
    if not os.path.exists(path):
        print(f"Weights Ctranspath not found.Downloading...")
        os.makedirs(os.path.dirname(path), exist_ok=True)
        url = "https://huggingface.co/jamesdolezal/CTransPath/resolve/main/ctranspath.pth"
        try:
            response = requests.get(url, stream=True)
            with open(path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(" Download Completed.")
        except Exception as e:
            print(f"Error download: {e}")

if MODEL_TO_RUN == "ctranspath":
    download_ctranspath_if_missing(CTRANSPATH_WEIGHTS)

# ==========================================

print(f"TRAINING ON BACH: {MODEL_TO_RUN.upper()}")



try:
    
    df = pd.read_csv(CSV_PATH, header=None, names=['filename', 'label_name'])
except:
    print("Error reading CSV.")


class_map = {'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3,
             'normal': 0, 'benign': 1, 'insitu': 2, 'invasive': 3}
df['label'] = df['label_name'].map(class_map)


df['base_name'] = df['filename'].apply(lambda x: os.path.splitext(x)[0])

patient_ids = df['base_name'].values
labels = df['label'].values


train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=labels)

print(f"Dataset Split:")
print(f"   -> Train: {len(train_ids)}")
print(f"   -> Val:   {len(val_ids)}")


class BachTileDataset(Dataset):
    def __init__(self, root_dir, valid_base_names, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_set = set(valid_base_names)
        


        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            
            for img_name in os.listdir(class_path):
                if not img_name.endswith('.tiff'): continue
                

                parts = img_name.split('_')
                if len(parts) >= 3:
                    base_name = "_".join(parts[:-2])
                else:
                    base_name = os.path.splitext(img_name)[0]
                
                
                if base_name in valid_set:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder))
                    


    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform: image = self.transform(image)
            return image, self.labels[idx]
        except: return torch.zeros(3, 224, 224), 0


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = BachTileDataset(TILES_DIR, train_ids, transform=train_transform)
val_dataset = BachTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


print(f"Configuration model: {MODEL_TO_RUN}...")

if MODEL_TO_RUN == "resnet":
    model = models.resnet50(weights='IMAGENET1K_V1') 
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

elif MODEL_TO_RUN == "ctranspath":
    def to_2tuple(x): return tuple(x) if isinstance(x, (tuple, list)) else (x, x)
    class ConvStem(nn.Module):
        def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
            super().__init__()
            img_size = to_2tuple(img_size)
            patch_size = to_2tuple(patch_size)
            self.stem = nn.Sequential(
                nn.Conv2d(in_chans, embed_dim // 2, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim // 2), nn.ReLU(inplace=True),
                nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim), nn.ReLU(inplace=True),
            )
        def forward(self, x):
            x = self.stem(x)
            x = x.permute(0, 2, 3, 1) 
            return x


    model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, embed_dim=128, depths=[2, 2, 18, 2], num_heads=[4, 8, 16, 32])
    model.patch_embed = ConvStem(img_size=224, patch_size=4, in_chans=3, embed_dim=128, norm_layer=nn.LayerNorm)


    checkpoint = torch.load(CTRANSPATH_WEIGHTS, map_location="cpu")
    if 'model' in checkpoint: checkpoint = checkpoint['model']
    model_dict = model.state_dict()
    valid_weights = {k: v for k, v in checkpoint.items() if k in model_dict and model_dict[k].shape == v.shape}
    model.load_state_dict(valid_weights, strict=False)

    model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    def pooling_forward(self, x):
        x = self.forward_features(x)
        x = x.mean(dim=[1, 2]) 
        x = self.head(x)
        return x
    model.forward = types.MethodType(pooling_forward, model)

elif MODEL_TO_RUN == "phikon":

    try:

        model = timm.create_model("hf_hub:owkin/phikon", pretrained=True, num_classes=NUM_CLASSES)
    except Exception as e:
        model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)

elif MODEL_TO_RUN == "uni":
    model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, num_classes=NUM_CLASSES)
    for param in model.parameters(): param.requires_grad = False # Freeze
    for param in model.head.parameters(): param.requires_grad = True

elif MODEL_TO_RUN == "virchow2":
    model = timm.create_model("hf-hub:paige-ai/Virchow2", pretrained=True, mlp_layer=timm.layers.SwiGLUPacked, act_layer=torch.nn.SiLU, num_classes=NUM_CLASSES)
    for param in model.parameters(): param.requires_grad = False
    for param in model.head.parameters(): param.requires_grad = True

elif MODEL_TO_RUN == "conch":
    import open_clip
    model_conch, _, _ = open_clip.create_model_and_transforms('conch_ViT-B-16', pretrained='hf-hub:MahmoodLab/CONCH', force_custom_clip=True)
    visual_model = model_conch.visual
    class CONCHWrapper(nn.Module):
        def __init__(self, backbone, num_classes):
            super().__init__()
            self.backbone = backbone
            for param in self.backbone.parameters(): param.requires_grad = False
            self.head = nn.Linear(512, num_classes)
        def forward(self, x): return self.head(self.backbone(x))
    model = CONCHWrapper(visual_model, NUM_CLASSES)

model = model.to(device)





all_labels = train_dataset.labels 

if len(all_labels) > 0:
    cw = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)
    print(f"   -> Pesi: {class_weights}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()


params_to_update = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(params_to_update, lr=LR)

print(f"\n START TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")


history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []} 
best_acc = 0.0

import time
import json 

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # Train
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # Val
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time 
    
    print(f"Epoch {epoch+1}: Loss={epoch_loss:.4f} | Acc={val_acc:.2f}% | Time={duration:.0f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(duration) 
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))



json_path = os.path.join(SAVE_DIR, f"stats_{MODEL_TO_RUN}.json")
with open(json_path, "w") as f:
    json.dump(history, f)
print(f"Statistics exported in: {json_path}")





UNI

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import timm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from huggingface_hub import login
import time
import json
import gc


gc.collect()
torch.cuda.empty_cache()



MODEL_TO_RUN = "uni"   

BATCH_SIZE = 4              
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 4             # BACH: Normal, Benign, InSitu, Invasive

# PERCORSI
TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv"
SAVE_DIR = "./risultati_finali_bach"


Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

print(f"TRAINING ON BACH: {MODEL_TO_RUN.upper()}")



try:
    df = pd.read_csv(CSV_PATH, header=None, names=['filename', 'label_name'])
except:
    print("Error reading CSV.")


class_map = {'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3,
             'normal': 0, 'benign': 1, 'insitu': 2, 'invasive': 3}
df['label'] = df['label_name'].map(class_map)
df['base_name'] = df['filename'].apply(lambda x: os.path.splitext(x)[0])

patient_ids = df['base_name'].values
labels = df['label'].values


train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=labels)

print(f"Dataset Split -> Train: {len(train_ids)} | Val: {len(val_ids)}")


class BachTileDataset(Dataset):
    def __init__(self, root_dir, valid_base_names, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_set = set(valid_base_names)
        

        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            for img_name in os.listdir(class_path):
                if not img_name.endswith('.tiff'): continue
                parts = img_name.split('_')
                if len(parts) >= 3:
                    base_name = "_".join(parts[:-2])
                else:
                    base_name = os.path.splitext(img_name)[0]
                
                if base_name in valid_set:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder))


    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform: image = self.transform(image)
            return image, self.labels[idx]
        except: return torch.zeros(3, 224, 224), 0

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = BachTileDataset(TILES_DIR, train_ids, transform=train_transform)
val_dataset = BachTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


print(f"Configuration model: {MODEL_TO_RUN.upper()}...")

try:
    print("   loading UNI from Hugging Face...")
    
    from timm.layers import SwiGLUPacked

    timm_kwargs = {
        'img_size': 224, 
        'patch_size': 14, 
        'depth': 24, 
        'num_heads': 24, 
        'init_values': 1e-5, 
        'embed_dim': 1536, 
        'mlp_ratio': 2.66667*2, 
        'num_classes': 0,       
        'no_embed_class': True, 
        'mlp_layer': SwiGLUPacked, 
        'act_layer': torch.nn.SiLU, 
        'reg_tokens': 8,        
        'dynamic_img_size': True
    }
    
    backbone = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs)
    

    
    class UNIWrapper(nn.Module):
        def __init__(self, backbone, num_classes):
            super().__init__()
            self.backbone = backbone
            # Freeze del backbone
            for param in self.backbone.parameters(): 
                param.requires_grad = False

            self.head = nn.Linear(1536, num_classes)
            
        def forward(self, x): 
            return self.head(self.backbone(x))
            
    model = UNIWrapper(backbone, NUM_CLASSES)
        
except Exception as e:
    print(f"Error UNI: {e}")
    raise

model = model.to(device)


all_labels = train_dataset.labels 

if len(all_labels) > 0:
    cw = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)
    print(f"   -> Pesi: {class_weights}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")

history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    epoch_time = time.time() - start_time
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    
    print(f"Epoch {epoch+1}: Loss={epoch_loss:.4f} | Acc={val_acc:.2f}% | Tempo={epoch_time:.0f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(epoch_time)
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))


with open(os.path.join(SAVE_DIR, f"stats_{MODEL_TO_RUN}.json"), 'w') as f:
    json.dump(history, f)





CONCH

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import gc
import json
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from huggingface_hub import login

gc.collect()
torch.cuda.empty_cache()


MODEL_TO_RUN = "conch"
BATCH_SIZE = 16      
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 4      # BACH: Normal, Benign, InSitu, Invasive

# PERCORSI BACH
TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv"
SAVE_DIR = "./risultati_finali_bach"


Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"TRAINING ON BACH: {MODEL_TO_RUN.upper()}")
print(f"Device: {device}")



try:
    df = pd.read_csv(CSV_PATH, header=None, names=['filename', 'label_name'])
except:
    print("Error reading CSV.")


class_map = {'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3,
             'normal': 0, 'benign': 1, 'insitu': 2, 'invasive': 3}
df['label'] = df['label_name'].map(class_map)
df['base_name'] = df['filename'].apply(lambda x: os.path.splitext(x)[0])

patient_ids = df['base_name'].values
labels = df['label'].values


train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=labels)

print(f"Dataset Split -> Train: {len(train_ids)} | Val: {len(val_ids)}")


class BachTileDataset(Dataset):
    def __init__(self, root_dir, valid_base_names, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_set = set(valid_base_names)
        

        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            for img_name in os.listdir(class_path):
                if not img_name.endswith('.tiff'): continue
                parts = img_name.split('_')
                if len(parts) >= 3:
                    base_name = "_".join(parts[:-2])
                else:
                    base_name = os.path.splitext(img_name)[0]
                
                if base_name in valid_set:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder))


    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform: image = self.transform(image)
            return image, self.labels[idx]
        except: return torch.zeros(3, 224, 224), 0

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = BachTileDataset(TILES_DIR, train_ids, transform=train_transform)
val_dataset = BachTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


print(f"Configuration model: {MODEL_TO_RUN.upper()}...")

try:
    import open_clip
    from open_clip import factory
    from huggingface_hub import hf_hub_download


    factory._MODEL_CONFIGS['conch_ViT-B-16'] = {
        "embed_dim": 512,
        "vision_cfg": {"image_size": 224, "layers": 12, "width": 768, "patch_size": 16},
        "text_cfg": {"context_length": 77, "vocab_size": 49408, "width": 512, "heads": 8, "layers": 12}
    }

    class CONCHClassifier(nn.Module):
        def __init__(self, num_classes):
            super().__init__()
  
            model, _, _ = open_clip.create_model_and_transforms('conch_ViT-B-16', pretrained=None)
            
 
            checkpoint_path = hf_hub_download(repo_id="MahmoodLab/CONCH", filename="pytorch_model.bin")
            checkpoint = torch.load(checkpoint_path, map_location='cpu')
            
            if 'state_dict' in checkpoint:
                checkpoint = checkpoint['state_dict']
            
            model.load_state_dict(checkpoint, strict=False)

            
            self.backbone = model.visual
            

            for param in self.backbone.parameters():
                param.requires_grad = False

            
            self.head = nn.Linear(512, num_classes)
            
        def forward(self, x):
            with torch.no_grad():
                features = self.backbone(x)
            if isinstance(features, tuple):
                features = features[0]
            return self.head(features)

    model = CONCHClassifier(NUM_CLASSES)



except Exception as e:
    print(f"ErrorCONCH: {e}")
    raise e

model = model.to(device)


all_train_labels = train_dataset.labels 

if len(all_train_labels) > 0:
    cw = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(model.head.parameters(), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # --- TRAIN ---
    model.train() 
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # --- VALIDATION ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}% | Time={duration:.1f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(duration)
    

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))


with open(os.path.join(SAVE_DIR, f"stats_{MODEL_TO_RUN}.json"), 'w') as f:
    json.dump(history, f)



VIRCHOW

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import timm
import gc
import json
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from huggingface_hub import login

gc.collect()
torch.cuda.empty_cache()


MODEL_TO_RUN = "virchow2"

BATCH_SIZE = 8       
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 4      # BACH: Normal, Benign, InSitu, Invasive


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv"
SAVE_DIR = "./risultati_finali_bach"


Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print(f"TRAINING ON BACH: {MODEL_TO_RUN.upper()}")
print(f"Device: {device}")



try:
    df = pd.read_csv(CSV_PATH, header=None, names=['filename', 'label_name'])
except:
    print("Error reading CSV.")


class_map = {'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3,
             'normal': 0, 'benign': 1, 'insitu': 2, 'invasive': 3}
df['label'] = df['label_name'].map(class_map)
df['base_name'] = df['filename'].apply(lambda x: os.path.splitext(x)[0])

patient_ids = df['base_name'].values
labels = df['label'].values


train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=labels)
print(f"Dataset Split -> Train: {len(train_ids)} | Val: {len(val_ids)}")


class BachTileDataset(Dataset):
    def __init__(self, root_dir, valid_base_names, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_set = set(valid_base_names)
        

        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            for img_name in os.listdir(class_path):
                if not img_name.endswith('.tiff'): continue
                parts = img_name.split('_')
                if len(parts) >= 3:
                    base_name = "_".join(parts[:-2])
                else:
                    base_name = os.path.splitext(img_name)[0]
                
                if base_name in valid_set:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder))


    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform: image = self.transform(image)
            return image, self.labels[idx]
        except: return torch.zeros(3, 224, 224), 0

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = BachTileDataset(TILES_DIR, train_ids, transform=train_transform)
val_dataset = BachTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Configuration model: {MODEL_TO_RUN.upper()}...")

class Virchow2Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        print("   ⏳ Caricamento Backbone Virchow2 da HuggingFace...")
        

        self.backbone = timm.create_model(
            "hf-hub:paige-ai/Virchow2", 
            pretrained=True, 
            mlp_layer=timm.layers.SwiGLUPacked, 
            act_layer=torch.nn.SiLU,
            num_classes=0,            
            dynamic_img_size=True     
        )
        

        for param in self.backbone.parameters():
            param.requires_grad = False

        

        self.head = nn.Linear(1280, num_classes)
        
    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
            

        if features.ndim == 3: 
            features = features.mean(dim=1) 
            
        return self.head(features)

try:
    model = Virchow2Classifier(NUM_CLASSES).to(device)

except Exception as e:
    print(f"Error loading": {e}")
    raise e


all_train_labels = train_dataset.labels
if len(all_train_labels) > 0:
    cw = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)
    print(f"   -> Pesi Classi calcolati: {class_weights}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.head.parameters(), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # --- TRAIN ---
    model.train() 
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        images = images.to(device)
        labels = labels.to(device).long()
        if labels.dim() > 1: labels = labels.squeeze()
        
        optimizer.zero_grad()
        outputs = model(images) 
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # --- VALIDATION ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            images = images.to(device)
            labels = labels.to(device).long()
            if labels.dim() > 1: labels = labels.squeeze()
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}% | Time={duration:.1f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(duration)
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))

with open(os.path.join(SAVE_DIR, f"stats_{MODEL_TO_RUN}.json"), 'w') as f:
    json.dump(history, f)

